# Bayesian Inference: Beta-Binomial Post-Test Analysis

**Method:** Bayesian inference using a Beta-Binomial conjugate model — estimates the posterior distribution of the true conversion rate given observed data.

---

$$\text{Prior: } \theta \sim \text{Beta}(\alpha_0, \beta_0) \quad \text{(uninformative: } \alpha_0 = \beta_0 = 1\text{)}$$

$$\text{Likelihood: } X | \theta \sim \text{Binomial}(n, \theta)$$

$$\text{Posterior: } \theta | X \sim \text{Beta}(\alpha_0 + x, \;\beta_0 + n - x)$$

---

| Symbol | Meaning |
|--------|---------|
| $\theta$ | True conversion rate (unknown parameter) |
| $\alpha_0, \beta_0$ | Prior hyperparameters (1, 1 = uniform / uninformative) |
| $n$ | Number of visitors (trials) |
| $x$ | Number of conversions (successes) |
| $\text{E}[\theta \| \text{data}]$ | Posterior mean = $\frac{\alpha_0 + x}{\alpha_0 + x + \beta_0 + n - x}$ |

## Data Input

Enter your observed data below.

In [48]:
import json

config = json.load(open("../test_config.json"))
d = config["design"]
planned_n = config["computed"]["planned_n"]
credible_threshold = 1 - d["alpha"]  # alpha=0.05 → 0.95

# ==========================================
#  OBSERVED DATA — enter your results here
# ==========================================
visitors    = 600    # unique visitors
conversions = 6      # unique conversions
# ==========================================

In [49]:
import numpy as np
import plotly.graph_objects as go
from scipy.stats import beta

def run_bayesian_post_test(visitors, conversions, target_rate, credible_threshold=0.95):
    post_alpha = 1 + conversions
    post_beta = 1 + (visitors - conversions)
    p_hat = conversions / visitors
    mean_rate = post_alpha / (post_alpha + post_beta)

    ci_tail = (1 - credible_threshold) / 2
    ci_lower = beta.ppf(ci_tail, post_alpha, post_beta)
    ci_upper = beta.ppf(1 - ci_tail, post_alpha, post_beta)
    decision_upper = beta.ppf(credible_threshold, post_alpha, post_beta)
    decision_lower = beta.ppf(1 - credible_threshold, post_alpha, post_beta)

    prob_above = 1 - beta.cdf(target_rate, post_alpha, post_beta)
    prob_below = beta.cdf(target_rate, post_alpha, post_beta)
    thresh_pct = f"{credible_threshold:.0%}"

    if prob_above >= credible_threshold:
        verdict, verdict_en = "続行", "PROCEED"
    elif prob_below >= credible_threshold:
        verdict, verdict_en = "中止", "STOP"
    else:
        verdict, verdict_en = "証拠不十分", "INSUFFICIENT EVIDENCE"

    # --- Output ---
    print(f"--- ベイズ推定 事後分析 ---")
    print(f"手法 : ベイズ推定（ベータ二項分布モデル）")
    print(f"KPI  : コンバージョン率（CVR）")
    print(f"目標 : {target_rate:.1%}")
    print(f"信用閾値: {thresh_pct}")
    print(f"-" * 50)
    print(f"【観測データ】")
    print(f"  訪問者数 (UU)  : {visitors:,}")
    print(f"  CV数 (UU)      : {conversions:,}")
    print(f"  実測CVR        : {p_hat:.4f} ({p_hat:.2%})")
    print(f"-" * 50)
    print(f"推定CVR : E[θ|data] = {mean_rate:.2%}")
    print(f"{thresh_pct} 信用区間 : [{ci_lower:.2%}, {ci_upper:.2%}]")
    print(f"P(θ > {target_rate:.1%}) = {prob_above:.1%}")
    print(f"P(θ < {target_rate:.1%}) = {prob_below:.1%}")
    print(f"-" * 50)
    print(f"判定 : {verdict_en} — {verdict}")
    print(f"-" * 50)

    # --- Plot ---
    x_max = max(mean_rate * 3, target_rate * 2, ci_upper * 1.5)
    x = np.linspace(0, x_max, 2000)
    y = beta.pdf(x, post_alpha, post_beta)
    y_max = max(y)

    fig = go.Figure()
    fig.add_trace(go.Scatter(x=x, y=y, mode='lines', name='P(θ|data) Posterior', line=dict(color='#1f77b4', width=3)))

    mask = (x >= ci_lower) & (x <= ci_upper)
    fig.add_trace(go.Scatter(x=x[mask], y=y[mask], mode='lines', fill='tozeroy',
        name=f'{thresh_pct} Credible Interval', line=dict(width=0), fillcolor='rgba(31,119,180,0.3)'))

    fig.add_vline(x=target_rate, line_width=3, line_dash="solid", line_color="#d62728")
    fig.add_vline(x=ci_lower, line_width=2, line_dash="dash", line_color="#1f77b4")
    fig.add_vline(x=ci_upper, line_width=2, line_dash="dash", line_color="#1f77b4")
    fig.add_vline(x=mean_rate, line_width=2, line_dash="dot", line_color="#005b96")
    fig.add_vline(x=decision_upper, line_width=2, line_dash="dashdot", line_color="#ff7f0e")
    fig.add_vline(x=decision_lower, line_width=2, line_dash="dashdot", line_color="#ff7f0e")

    fig.add_annotation(x=mean_rate, y=y_max*1.08, text=f"E[θ|data] = {mean_rate:.2%}", showarrow=False, yshift=10,
        font=dict(color="#005b96", size=12, weight='bold'), bgcolor="rgba(255,255,255,0.85)", bordercolor="#005b96", borderwidth=1, borderpad=3)
    fig.add_annotation(x=target_rate, y=y_max*0.88, text=f"Target: {target_rate:.1%}", showarrow=True, arrowhead=2, arrowcolor="#d62728",
        ax=-80, ay=-25, font=dict(color="#d62728", size=12, weight='bold'), bgcolor="rgba(255,255,255,0.85)", bordercolor="#d62728", borderwidth=1, borderpad=3)
    fig.add_annotation(x=decision_upper, y=y_max*0.68, text=f"θ_{credible_threshold:.0%} = {decision_upper:.2%}", showarrow=True, arrowhead=2, arrowcolor="#ff7f0e",
        ax=90, ay=-10, font=dict(color="#ff7f0e", size=10), bgcolor="rgba(255,255,255,0.85)", bordercolor="#ff7f0e", borderwidth=1, borderpad=3)
    fig.add_annotation(x=decision_lower, y=y_max*0.68, text=f"θ_{1-credible_threshold:.0%} = {decision_lower:.2%}", showarrow=True, arrowhead=2, arrowcolor="#ff7f0e",
        ax=-90, ay=-10, font=dict(color="#ff7f0e", size=10), bgcolor="rgba(255,255,255,0.85)", bordercolor="#ff7f0e", borderwidth=1, borderpad=3)

    fig.update_layout(
        title=dict(text=f"<b>Bayesian Post-Test Analysis</b><br>"
            f"P(θ|data) ~ Beta(α={post_alpha}, β={post_beta}) | P(θ > {target_rate:.1%}) = {prob_above:.1%} → <b>{verdict_en}</b>", font=dict(size=20)),
        xaxis_title="θ (Conversion Rate)", yaxis_title="P(θ|data)",
        xaxis=dict(tickformat=".1%", showgrid=True, gridcolor='#eee'), yaxis=dict(showgrid=True, gridcolor='#eee'),
        template="plotly_white", legend=dict(y=0.99, x=0.99, bgcolor="rgba(255,255,255,0.8)"), plot_bgcolor='rgba(0,0,0,0)')
    fig.show()

In [50]:
def sequential_check_bayesian(visitors, conversions, target_rate, planned_n=None, credible_threshold=0.95):
    """Bayesian sequential check — no alpha spending needed, check at any time."""
    post_alpha = 1 + conversions
    post_beta = 1 + (visitors - conversions)
    mean_rate = post_alpha / (post_alpha + post_beta)
    prob_above = 1 - beta.cdf(target_rate, post_alpha, post_beta)
    prob_below = beta.cdf(target_rate, post_alpha, post_beta)
    thresh_pct = f"{credible_threshold:.0%}"

    print(f"--- SEQUENTIAL CHECK (Bayesian) ---")
    if planned_n:
        print(f"進捗 : {visitors:,} / {planned_n:,} ({visitors/planned_n*100:.1f}%)")
    else:
        print(f"現在 : {visitors:,}人")
    print(f"実測CVR : {conversions}/{visitors} = {conversions/visitors:.2%}  |  推定CVR : {mean_rate:.2%}")
    print(f"P(θ > {target_rate:.1%}) = {prob_above:.1%}  |  P(θ < {target_rate:.1%}) = {prob_below:.1%}")
    print(f"-" * 50)

    if prob_above >= credible_threshold:
        print(f"判定 : PROCEED — P(θ>{target_rate:.1%}) = {prob_above:.1%} ≥ {thresh_pct}。データ収集不要")
    elif prob_below >= credible_threshold:
        print(f"判定 : STOP — P(θ<{target_rate:.1%}) = {prob_below:.1%} ≥ {thresh_pct}。データ収集不要")
    else:
        print(f"判定 : INSUFFICIENT — どちらも{thresh_pct}未達。データ収集を継続")
        if planned_n and (remaining := planned_n - visitors) > 0:
            print(f"         残り約 {remaining:,}人")
    print(f"-" * 50)

In [51]:
run_bayesian_post_test(visitors=visitors, conversions=conversions,
    target_rate=d["target_rate"], credible_threshold=credible_threshold)

print()
sequential_check_bayesian(visitors=visitors, conversions=conversions,
    target_rate=d["target_rate"], planned_n=planned_n,
    credible_threshold=credible_threshold)

--- ベイズ推定 事後分析 ---
手法 : ベイズ推定（ベータ二項分布モデル）
KPI  : コンバージョン率（CVR）
目標 : 2.0%
信用閾値: 95%
--------------------------------------------------
【観測データ】
  訪問者数 (UU)  : 600
  CV数 (UU)      : 6
  実測CVR        : 0.0100 (1.00%)
--------------------------------------------------
推定CVR : E[θ|data] = 1.16%
95% 信用区間 : [0.47%, 2.16%]
P(θ > 2.0%) = 4.4%
P(θ < 2.0%) = 95.6%
--------------------------------------------------
判定 : STOP — 中止
--------------------------------------------------



--- SEQUENTIAL CHECK (Bayesian) ---
進捗 : 600 / 1,283 (46.8%)
実測CVR : 6/600 = 1.00%  |  推定CVR : 1.16%
P(θ > 2.0%) = 4.4%  |  P(θ < 2.0%) = 95.6%
--------------------------------------------------
判定 : STOP — P(θ<2.0%) = 95.6% ≥ 95%。データ収集不要
--------------------------------------------------
